In [8]:
from lcb_runner.benchmarks import code_generation

dataset = code_generation.load_code_generation_dataset(release_version="release_v5", start_date="2024-01-01", diffulty=code_generation.Difficulty.HARD)

Using 'test' split of the dataset.
Filtered problems by difficulty: hard
Loaded 194 problems


In [9]:
from common.llm_client import create_llm_client, LLMClient
from common.code_processor import CodeProcessor

code_processor: CodeProcessor = CodeProcessor()
sample_problem: str = dataset[0].question_content
starter_code: str = dataset[0].starter_code 
llm_client: LLMClient  = create_llm_client(provider='openai', model='gpt-5')
print(f"LLM Reasoning Effort: {llm_client.reasoning_effort}")

print("Sample Problem: ")
print(sample_problem)

print("Starter Code: ")
print(starter_code)

print("=== Generating Code ===")

INITIAL_CODE_POPULATION_SIZE = 10
initial_code_prompt = f"Write {INITIAL_CODE_POPULATION_SIZE} distinct solutions to solve the following problem: {sample_problem}\n```python\n{starter_code}```. Each solution should be in a separate code block."
response = llm_client.generate(initial_code_prompt)


# formatting response
code_blocks = code_processor.extract_all_code_blocks(response)

if len(code_blocks) != INITIAL_CODE_POPULATION_SIZE:
    raise ValueError(f"Expected {INITIAL_CODE_POPULATION_SIZE} code blocks, but got {len(code_blocks)}\nResponse: {response}")


for code_idx, code in enumerate(code_blocks):
    print(f"Code Block {code_idx+1}:\n{code}\n")

LLM Reasoning Effort: minimal
Sample Problem: 
You are given a 0-indexed string s and an integer k.
You are to perform the following partitioning operations until s is empty:

Choose the longest prefix of s containing at most k distinct characters.
Delete the prefix from s and increase the number of partitions by one. The remaining characters (if any) in s maintain their initial order.

Before the operations, you are allowed to change at most one index in s to another lowercase English letter.
Return an integer denoting the maximum number of resulting partitions after the operations by optimally choosing at most one index to change.
 
Example 1:

Input: s = "accca", k = 2
Output: 3
Explanation: In this example, to maximize the number of resulting partitions, s[2] can be changed to 'b'.
s becomes "acbca".
The operations can now be performed as follows until s becomes empty:
- Choose the longest prefix containing at most 2 distinct characters, "acbca".
- Delete the prefix, and s becomes 

In [10]:
INITIAL_TEST_POPULATION_SIZE = 20
print("=== Generating Tests ===")
initial_test_prompt = f"Write {INITIAL_TEST_POPULATION_SIZE} distinct unit tests in Python for the following problem:\n{sample_problem}\nThe solution is imported in the format:\n{dataset[0].starter_code}\nWrite the tests using unittest framework. Do not use the examples givn in the problem."
print(initial_test_prompt)

response = llm_client.generate(initial_test_prompt)
print(f"response:\n{response}\n")
# formatting response
test_block = code_processor.extract_class_block(response)

if not test_block:
    raise ValueError(f"Expected a test class block, but none found.\nResponse: {response}")

print(f"Test Block:\n{test_block}\n")

=== Generating Tests ===
Write 20 distinct unit tests in Python for the following problem:
You are given a 0-indexed string s and an integer k.
You are to perform the following partitioning operations until s is empty:

Choose the longest prefix of s containing at most k distinct characters.
Delete the prefix from s and increase the number of partitions by one. The remaining characters (if any) in s maintain their initial order.

Before the operations, you are allowed to change at most one index in s to another lowercase English letter.
Return an integer denoting the maximum number of resulting partitions after the operations by optimally choosing at most one index to change.
 
Example 1:

Input: s = "accca", k = 2
Output: 3
Explanation: In this example, to maximize the number of resulting partitions, s[2] can be changed to 'b'.
s becomes "acbca".
The operations can now be performed as follows until s becomes empty:
- Choose the longest prefix containing at most 2 distinct characters, 

In [14]:
from common.sandbox import create_safe_test_environment
import numpy as np

matrix = np.zeros((INITIAL_CODE_POPULATION_SIZE, INITIAL_TEST_POPULATION_SIZE), dtype=int)
sandbox = create_safe_test_environment()

for code_idx, code in enumerate(code_blocks):
    print(f"Executing Code Block {code_idx+1}: ")
    script = code_processor.build_test_script_for_class(code, test_block)
    test_results = sandbox.execute_test_script(script)
    for test_idx, test in enumerate(test_results.test_results):
        print(f"Test: {test.name}, Status: {test.status}, Details: {test.details}")
        if test.status == "passed":
            matrix[code_idx, test_idx] = 1
        elif test.status == "failed":
            matrix[code_idx, test_idx] = 0  # failed
        else:
            matrix[code_idx, test_idx] = 0  # Let's treat errors as failures for now

Executing Code Block 1: 
Test: test_all_same_k1, Status: failed, Details: self.assertEqual(self.sol.maxPartitionsAfterOperations("aaaaaa", 1), 6)
AssertionError: 1 != 6
Test: test_all_same_k2, Status: passed, Details: None
Test: test_alternating_two_k1, Status: passed, Details: None
Test: test_alternating_two_k2, Status: passed, Details: None
Test: test_increasing_k1, Status: passed, Details: None
Test: test_increasing_k2, Status: passed, Details: None
Test: test_k1_change_helps, Status: failed, Details: self.assertEqual(self.sol.maxPartitionsAfterOperations("aabb", 1), 4)
AssertionError: 2 != 4
Test: test_k2_no_benefit, Status: passed, Details: None
Test: test_k_26, Status: passed, Details: None
Test: test_k_larger_than_distinct, Status: passed, Details: None
Test: test_k_plus_one_distinct, Status: passed, Details: None
Test: test_long_blocks_k1, Status: failed, Details: self.assertEqual(self.sol.maxPartitionsAfterOperations("aaaabbbbcccc", 1), 12)
AssertionError: 3 != 12
Test: test_l

In [15]:
print(matrix)

[[0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1]
 [0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1]
 [0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1]
 [0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1]
 [0 1 0 1 0 1 0 1 1 1 0 0 0 0 1 1 0 1 1 0]
 [0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1]
 [0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1]
 [0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1]
 [0 1 1 0 1 1 0 0 1 1 1 0 0 1 0 0 1 1 1 1]
 [0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1]]


In [35]:
from common.coevolution_bayesian import CoevolutionConfig, initialize_populations, update_population_beliefs, log_odds_array_to_probabilities

config = CoevolutionConfig(
    initial_code_population_size=INITIAL_CODE_POPULATION_SIZE,
    initial_test_population_size=INITIAL_TEST_POPULATION_SIZE,
    c0_prior=0.6,
    t0_prior=0.6
)

# Updated to match the new function signature (returns only 2 arrays)
code_log_odds, test_log_odds = initialize_populations(config)
new_code_log_odds, new_test_log_odds = update_population_beliefs(code_log_odds, test_log_odds, matrix)


new_code_probabilities = log_odds_array_to_probabilities(new_code_log_odds)
new_test_probabilities = log_odds_array_to_probabilities(new_test_log_odds)

print("=== Updated Probabilities ===")
print("Code:")
print([round(a, 2) for a in new_code_probabilities.tolist()])

print("Test:")
print([round(a, 2) for a in new_test_probabilities.tolist()])


print("\n\n=== Comparison with Pass Rates ===")
print("Pass rates for each code snippet:")
# in matrix rows are code snippets and columns are tests, so we take mean across axis 1
pass_rates = matrix.mean(axis=1)
print([round(a, 2) for a in pass_rates.tolist()])

print("Pass rates for each test:")
print([round(a, 2) for a in matrix.mean(axis=0).tolist()])

=== Updated Probabilities ===
Code:
[0.99, 0.99, 0.99, 0.99, 0.6, 0.99, 0.99, 0.99, 0.88, 0.99]
Test:
[0.03, 0.99, 0.97, 0.97, 0.97, 0.99, 0.03, 0.97, 0.99, 0.99, 0.97, 0.03, 0.94, 0.06, 0.06, 0.97, 0.97, 0.99, 0.99, 0.97]


=== Comparison with Pass Rates ===
Pass rates for each code snippet:
[0.75, 0.75, 0.75, 0.75, 0.5, 0.75, 0.75, 0.75, 0.6, 0.75]
Pass rates for each test:
[0.0, 1.0, 0.9, 0.9, 0.9, 1.0, 0.0, 0.9, 1.0, 1.0, 0.9, 0.0, 0.8, 0.1, 0.1, 0.9, 0.9, 1.0, 1.0, 0.9]
